# **Single-output Use Case Demonstration**


In [ ]:
import numpy as np
import pandas as pd
import random
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization
from keras.initializers import he_normal
from keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler

In [ ]:
# This function generates some synthetic data {sqft, bedrooms, lotsize and price}
def generate_data(count, noutputs=1):
    sqft_ranges = [0, 1500, 2500, 3500, 10000]
    price_ranges = [800000, 1000000, 2500000, 3000000]

    x = []
    y = []

    for i in range(count):
        sqft = random.randint(1800,4500)
        nbedrooms = 1+ int(sqft/1000)
        lotsize = int(random.uniform(1.2, 2.0) * sqft)

        price = 0
        for r in range(len(sqft_ranges)-1):
            if sqft >= sqft_ranges[r] and sqft < sqft_ranges[r+1]:
                price = price_ranges[r]
                break
        assert price != 0

        lotsize_value = int(random.randint(50000, 300000) * (lotsize/sqft))
        price = price + lotsize_value
        price = round(price, -3)

        if noutputs == 1:
            x.append([sqft, nbedrooms, lotsize])
            y.append(price)
        else:
            x.append([sqft, lotsize])
            y.append([price, nbedrooms])

    return x, y

# **Single-output Regression Use Case**


---

**Given the `sqft`, `bedrooms` and `lotsize` predict the `price` of a house**

**Let's initialize a Standardization (Normalization) module that we'll use in our model**

In [ ]:
norm_x = StandardScaler() # We'll use this to normalize our input
norm_y = StandardScaler() # We'll use this to normalize our output

**Let's `build` a model to predict 1 output "`price`", given a house's "`sqft`", "`lotsize`" and "`bedrooms`"**

In [ ]:
def build_model_for_one_output():
    model = Sequential()

    # Hidden Layer(s)
    model.add(Dense(10, input_dim=3, kernel_initializer=he_normal(), activation="relu"))

    # Output Layer
    model.add(Dense(1))

    model.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
    return model

model1 = build_model_for_one_output()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Let's generate some data**

In [ ]:
# Generate some house price data
x, y = generate_data(10000, noutputs=1)

# Let's display the data we generated
data = pd.DataFrame(
    {'sqft': [rec[0] for rec in x],
    'nbedrooms': [rec[1] for rec in x],
    'lotsize': [rec[2] for rec in x],
    'price': y})
data

,sqft,nbedrooms,lotsize,price
0,2986,3,4348,2803000
1,3226,4,6096,2814000
2,3097,4,4799,2726000
3,4308,5,7352,3400000
4,3830,4,4828,3084000
...,...,...,...,...
9995,4068,5,5529,3222000
9996,3792,4,6736,3358000
9997,2545,3,4824,2680000
9998,3731,4,6646,3367000


**Let's `train` our model**

In [ ]:
def train_model_for_one_output(model, x, y):
    # Normalize input data
    xnorm = norm_x.fit_transform(np.array(x))
    ynorm = norm_y.fit_transform(np.array(y).reshape(-1,1))

    # Train the model
    model.fit(xnorm, ynorm, epochs=100, batch_size=10, verbose=1)

train_model_for_one_output(model1, x, y)

Epoch 1/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1305
Epoch 2/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1053
Epoch 3/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0966
Epoch 4/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0846
Epoch 5/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0807
Epoch 6/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0765
Epoch 7/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0768
Epoch 8/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0748
Epoch 9/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0742
Epoch 10/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0727
Epoch 11/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0741
Epoch 12/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0728
Epoch 13/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0731
Epoch 14/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0733
Epoch 15/100
10

**Let's use our model to make predictions**

In [ ]:
def predict_model_for_one_output(model):
    x, yt = generate_data(20, noutputs=1)

    # We have to normalize the input again even during prediction because we normalized during training
    xnorm = norm_x.transform(np.array(x))

    predictions = model.predict(xnorm)

    # De-Normalize predictions
    yp = norm_y.inverse_transform(predictions).flatten()

    return x, yt, yp

x, yt, yp = predict_model_for_one_output(model1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


**Let's see how well our model did with the predictions**

In [ ]:
def show_predictions_for_one_output(x, yt, yp):
    # Pretty Print the Predictions comparing them what they should be
    offby = []
    percent = []
    for i in range(len(yp)):
        diff = int(yt[i] - yp[i])
        offby.append(diff)
        percent.append(int(100*(diff/yt[i])))

    data = pd.DataFrame(
        {'sqft': [rec[0] for rec in x],
            'nbedrooms': [rec[1] for rec in x],
            'lotsize': [rec[2] for rec in x],
            'price': yt,
            'price_pred': [int(p) for p in yp],
            'price_offby': offby,
            'price_percent_off': percent})
    return data

show_predictions_for_one_output(x, yt, yp)

,sqft,nbedrooms,lotsize,price,price_pred,price_offby,price_percent_off
0,2602,3,3392,2584000,2712678,-128678,-4
1,3924,4,7792,3476000,3340369,135631,3
2,3165,4,5935,2817000,2780837,36162,1
3,4203,5,6836,3449000,3221009,227990,6
4,3180,4,6227,2767000,2791874,-24874,0
5,2381,3,3468,1217000,1265414,-48414,-3
6,3577,4,6643,3519000,3291307,227693,6
7,4024,5,5375,3257000,3171503,85496,2
8,3052,4,5149,2622000,2755122,-133122,-5
9,3312,4,4028,2693000,2695292,-2292,0


**Let's initialize a Standardization module we'll use in our model**

In [ ]:
norm_x = StandardScaler() # We'll use this to standardize our input
norm_y = StandardScaler() # We'll use this to standardize our output

**Let's `build` a model to predict 2 outputs**

In [ ]:
def build_model_for_two_outputs():
    model = Sequential()

    # Hidden Layer(s)
    model.add(Dense(10, input_dim=2, kernel_initializer=he_normal(), activation="relu"))

    # Output Layer
    model.add(Dense(2))

    model.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
    return model

model2 = build_model_for_two_outputs()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Let's generate some data**

In [ ]:

# Generate some house price data
x, y = generate_data(10000, noutputs=2)

# Let's display the data we generated
data = pd.DataFrame(
    {'X_sqft': [rec[0] for rec in x],
    'X_lotsize': [rec[1] for rec in x],
    'Y_price': [rec[0] for rec in y],
    'Y_nbedrooms': [rec[1] for rec in y]})
data

,X_sqft,X_lotsize,Y_price,Y_nbedrooms
0,2080,3796,1324000,3
1,2537,3855,2875000,3
2,3392,4512,2881000,4
3,4206,7218,3417000,5
4,3884,7664,3364000,4
...,...,...,...,...
9995,2103,3749,1531000,3
9996,3974,5863,3349000,4
9997,2655,3534,2780000,3
9998,2349,3804,1311000,3


**Let's `train` our model**

In [ ]:
def train_model_for_two_outputs(model, x, y):
    # Normalize data
    xnorm = norm_x.fit_transform(np.array(x))
    ynorm = norm_y.fit_transform(np.array(y))

    # Train the model
    model.fit(xnorm, ynorm, epochs=100, batch_size=10, verbose=1)

train_model_for_two_outputs(model2, x, y)

Epoch 1/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1429
Epoch 2/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1315
Epoch 3/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1257
Epoch 4/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1191
Epoch 5/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0897
Epoch 6/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0735
Epoch 7/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0723
Epoch 8/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0702
Epoch 9/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0709
Epoch 10/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0693
Epoch 11/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0678
Epoch 12/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0681
Epoch 13/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0677
Epoch 14/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0670
Epoch 15/100
10

**Let's use our model to make predictions**

In [ ]:
def predict_model_for_two_outputs(model):
    x, yt = generate_data(20, noutputs=2)

    xnorm = norm_x.transform(np.array(x))

    predictions = model.predict(xnorm)

    # De-Normalize predictions
    yp = norm_y.inverse_transform(predictions)

    return x, yt, yp

x, yt, yp = predict_model_for_two_outputs(model2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step


**Let's see what our predictions look like**

In [ ]:
def show_predictions_for_two_outputs(x, yt, yp):
    price_offby = []
    price_percent = []

    nbedrooms_offby = []
    nbedrooms_percent = []

    for i in range(len(yp)):
        price_diff = int(yt[i][0] - yp[i][0])
        price_offby.append(price_diff)
        price_percent.append(int(100*(price_diff/yt[i][0])))

        nbedrooms_diff = yt[i][1] - yp[i][1]
        nbedrooms_offby.append(nbedrooms_diff)
        nbedrooms_percent.append(int(100*(nbedrooms_diff/yt[i][1])))

    data = pd.DataFrame(
            {'sqft': [rec[0] for rec in x],
            'lotsize': [rec[1] for rec in x],
            'price': [int(p[0]) for p in yt],
            'price_pred': [int(p[0]) for p in yp],
            'price_offby': price_offby,
            'price_percent_off': price_percent,
            'nbedrooms': [p[1] for p in yt],
            'nbedrooms_pred': [int(p[1]) for p in yp],
            'nbedrooms_offby': nbedrooms_offby,
            'nbedrooms_percent_of': nbedrooms_percent})
    return data

show_predictions_for_two_outputs(x, yt, yp)

,sqft,lotsize,price,price_pred,price_offby,price_percent_off,nbedrooms,nbedrooms_pred,nbedrooms_offby,nbedrooms_percent_of
0,2999,5574,2629000,2778711,-149711,-5,3,3,-0.485880,-16
1,4056,7807,3356000,3339086,16913,0,5,4,0.484668,9
2,2908,5448,2677000,2829505,-152505,-5,3,3,-0.008529,0
3,3035,5412,2952000,2736808,215192,7,4,3,0.286586,7
4,4476,8836,3435000,3340485,94514,2,5,5,-0.237846,-4
5,1952,3002,1260000,1304366,-44366,-3,2,2,-0.509751,-25
6,3648,7002,3186000,3250715,-64715,-2,4,3,0.012584,0
7,3239,5312,2714000,2827090,-113090,-4,4,3,0.020123,0
8,3836,6722,3418000,3316634,101365,2,4,4,-0.126339,-3
9,2286,3258,1368000,1244803,123196,9,3,3,-0.043181,-1
